In [ ]:
import panel as pn
import pandas as pd
from config.database import SessionLocal
from models import Pagamento
from datetime import date

pn.extension('tabulator', design='material', sizing_mode="stretch_width")

custom_css = """
.premium-btn {
    border-radius: 6px !important;
    font-weight: 600 !important;
    letter-spacing: 0.5px !important;
    transition: all 0.2s ease-in-out !important;
}
.premium-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(0,0,0,0.15) !important;
}
.premium-card-title {
    font-size: 1.1rem !important;
    color: #1e293b !important;
    font-weight: 700 !important;
}
"""
pn.config.raw_css.append(custom_css)

status_alert = pn.pane.Alert("", alert_type="light", visible=False, sizing_mode="stretch_width")

txt_id = pn.widgets.TextInput(name="ID Pagamento", placeholder="Deixe vazio para novo", disabled=False, sizing_mode="stretch_width")
txt_cpf_resp = pn.widgets.TextInput(name="CPF do Responsável", placeholder="Ex: 111.111.111-11", sizing_mode="stretch_width")
txt_cpf_cuid = pn.widgets.TextInput(name="CPF do Cuidador", placeholder="Ex: 222.222.222-22", sizing_mode="stretch_width")
num_valor = pn.widgets.FloatInput(name="Valor do Serviço (R$)", value=0.0, step=10.0, sizing_mode="stretch_width")

btn_salvar = pn.widgets.Button(name="Lançar / Atualizar", button_type="primary", icon="device-floppy", css_classes=['premium-btn'])
btn_excluir = pn.widgets.Button(name="Excluir Selecionado", button_type="danger", icon="trash", css_classes=['premium-btn'])
btn_limpar = pn.widgets.Button(name="Limpar Formulário", button_type="light", icon="eraser", css_classes=['premium-btn'])

tabela_dados = pn.widgets.Tabulator(
    pd.DataFrame(),
    height=320,
    layout='fit_columns',
    selectable='checkbox',
    theme='fast',
    pagination='remote',
    page_size=7
)

def carregar_dados():
    db = SessionLocal()
    try:
        query = db.query(Pagamento).order_by(Pagamento.id_pagamento.desc()).all()
        dados = [{
            "ID": p.id_pagamento,
            "CPF Responsável": p.cpf_responsavel,
            "CPF Cuidador": p.cpf_cuidador,
            "Valor (R$)": float(p.valor) if p.valor else 0.0,
            "Data": p.data_pagamento
        } for p in query]
        tabela_dados.value = pd.DataFrame(dados)
    finally:
        db.close()

@pn.depends(tabela_dados.param.selection, watch=True)
def preencher_formulario(selection):
    if not selection:
        limpar_campos_sem_limpar_selecao()
        return
    linha_selecionada = tabela_dados.value.iloc[selection[0]]
    txt_id.value = str(linha_selecionada['ID'])
    txt_cpf_resp.value = str(linha_selecionada['CPF Responsável']) if pd.notna(linha_selecionada['CPF Responsável']) else ""
    txt_cpf_cuid.value = str(linha_selecionada['CPF Cuidador']) if pd.notna(linha_selecionada['CPF Cuidador']) else ""
    num_valor.value = float(linha_selecionada['Valor (R$)']) if pd.notna(linha_selecionada['Valor (R$)']) else 0.0

def limpar_campos_sem_limpar_selecao():
    txt_id.value = ""
    txt_cpf_resp.value = ""
    txt_cpf_cuid.value = ""
    num_valor.value = 0.0

def limpar_campos(event):
    limpar_campos_sem_limpar_selecao()
    tabela_dados.selection = []

def salvar_registro(event):
    if not txt_cpf_resp.value or not txt_cpf_cuid.value:
        status_alert.object = "Erro: CPFs do Responsável e Cuidador são obrigatórios!"
        status_alert.alert_type = "danger"
        status_alert.visible = True
        return

    db = SessionLocal()
    try:
        pagamento = None
        if txt_id.value.isdigit():
            pagamento = db.query(Pagamento).filter_by(id_pagamento=int(txt_id.value)).first()

        if not pagamento:
            pagamento = Pagamento(
                cpf_responsavel=txt_cpf_resp.value,
                cpf_cuidador=txt_cpf_cuid.value,
                valor=num_valor.value,
                data_pagamento=date.today()
            )
            db.add(pagamento)
            msg = "Pagamento lançado com sucesso!"
        else:
            pagamento.cpf_responsavel = txt_cpf_resp.value
            pagamento.cpf_cuidador = txt_cpf_cuid.value
            pagamento.valor = num_valor.value
            msg = "Pagamento atualizado com sucesso!"

        db.commit()
        status_alert.object = msg
        status_alert.alert_type = "success"
        status_alert.visible = True
        carregar_dados()
        limpar_campos(None)
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro de banco de dados: Verifique se os CPFs informados já existem nas outras tabelas. ({str(e)})"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

def excluir_registro(event):
    selecao = tabela_dados.selection
    if not selecao:
        status_alert.object = "Selecione um pagamento na tabela para excluir."
        status_alert.alert_type = "warning"
        status_alert.visible = True
        return

    id_selecionado = tabela_dados.value.iloc[selecao[0]]['ID']
    db = SessionLocal()
    try:
        pagamento = db.query(Pagamento).filter_by(id_pagamento=int(id_selecionado)).first()
        if pagamento:
            db.delete(pagamento)
            db.commit()
            status_alert.object = "Registro de pagamento excluído com sucesso!"
            status_alert.alert_type = "success"
            status_alert.visible = True
            carregar_dados()
            limpar_campos(None)
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro ao excluir: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

btn_salvar.on_click(salvar_registro)
btn_excluir.on_click(excluir_registro)
btn_limpar.on_click(limpar_campos)

formulario_card = pn.Card(
    pn.Column(
        pn.Row(txt_id, txt_cpf_resp, txt_cpf_cuid),
        pn.Row(num_valor),
        pn.layout.Divider(),
        pn.Row(btn_salvar, btn_excluir, btn_limpar)
    ),
    title="💸 Lançamento de Pagamentos",
    header_css_classes=['premium-card-title']
)

tabela_card = pn.Card(
    tabela_dados,
    title="📊 Histórico de Transações",
    header_css_classes=['premium-card-title']
)

btn_pacientes = pn.widgets.Button(
    name="Módulo Pacientes",
    button_type="light",
    sizing_mode="stretch_width"
)
btn_pacientes.js_on_click(code="window.location.href='/'")

btn_cuidadores = pn.widgets.Button(
    name="Módulo Cuidadores",
    button_type="light",
    sizing_mode="stretch_width"
)
btn_cuidadores.js_on_click(code="window.location.href='/tela_cuidador'")

btn_transacoes = pn.widgets.Button(
    name="Módulo Transações",
    button_type="primary",
    sizing_mode="stretch_width",
    disabled=True
)

sidebar_content = pn.Column(
    pn.pane.Markdown("### 🏥 Menu Principal", margin=(0, 0, 10, 0)),
    btn_pacientes,
    btn_cuidadores,
    btn_transacoes,
    pn.layout.Divider(),
    pn.pane.Markdown(
        "**Usuário:** Admin\n\n**Status:** Online 🟢",
        styles={'color': '#cbd5e1', 'font-size': '13px'}
    ),
    sizing_mode="stretch_width"
)

template = pn.template.FastListTemplate(
    title='CareLink | Hub de Saúde',
    header_background="#0f172a",
    accent_base_color="#3b82f6",
    sidebar=[sidebar_content],
    main=[
        pn.Column(
            status_alert,
            formulario_card,
            pn.Spacer(height=15),
            tabela_card,
            sizing_mode="stretch_width"
        )
    ]
)

carregar_dados()
template.servable()